Lamar Alhimali S23108185

LAB3

Data Preprocessing & Feature Engineering

In [1]:
import pandas as pd
import numpy as np

PART 1: Create Messy Data

In [2]:
np.random.seed(42)

n = 200
data = {
    'age': np.random.randint(18, 80, n).astype(float),
    'blood_pressure': np.round(np.random.uniform(90, 180, n), 1),
    'cholesterol': np.round(np.random.uniform(150, 350, n), 1),
    'bmi': np.round(np.random.uniform(18, 42, n), 1),
    'gender': np.random.choice(['Male', 'Female'], n),
    'city': np.random.choice(['Jeddah', 'Riyadh', 'Dammam', 'Makkah'], n),
    'smoker': np.random.choice(['Yes', 'No'], n, p=[0.3, 0.7]),
    'heart_disease': np.random.choice([0, 1], n, p=[0.6, 0.4])
}

df = pd.DataFrame(data)

# Add missing values
missing_idx = np.random.choice(n, 20, replace=False)
df.loc[missing_idx[:10], 'age'] = np.nan
df.loc[missing_idx[10:15], 'blood_pressure'] = np.nan
df.loc[missing_idx[15:], 'cholesterol'] = np.nan

df.head()

,age,blood_pressure,cholesterol,bmi,gender,city,smoker,heart_disease
0,56.0,171.7,295.2,34.4,Male,Riyadh,No,1
1,69.0,112.4,345.2,21.9,Female,Dammam,No,0
2,46.0,126.9,253.3,39.9,Female,Jeddah,No,0
3,32.0,158.0,214.6,37.7,Female,Riyadh,No,1
4,60.0,110.6,309.0,40.8,Male,Makkah,Yes,1


Step 2: Explore

In [3]:
df.info()
df.isnull().sum()
(df.isnull().sum() / len(df)) * 100
df['city'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             190 non-null    float64
 1   blood_pressure  195 non-null    float64
 2   cholesterol     195 non-null    float64
 3   bmi             200 non-null    float64
 4   gender          200 non-null    object 
 5   city            200 non-null    object 
 6   smoker          200 non-null    object 
 7   heart_disease   200 non-null    int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 12.6+ KB


,count
city,
Dammam,59
Jeddah,52
Riyadh,47
Makkah,42


PART 2: Fix Missing Values

Step 1: Impute

In [4]:

from sklearn.impute import SimpleImputer

numeric_cols = ['age', 'blood_pressure', 'cholesterol']

imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

df.isnull().sum()

,0
age,0
blood_pressure,0
cholesterol,0
bmi,0
gender,0
city,0
smoker,0
heart_disease,0


Task 2 (Experiment)

In [6]:
imputer = SimpleImputer(strategy='mean')

Mean vs Median difference
Dropping rows loses data

PART 3: Encoding (TEXT → NUMBERS)


Step 1: Label Encoding

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])
df['smoker_encoded'] = le.fit_transform(df['smoker'])

Step 2: One-Hot Encoding

In [8]:
city_dummies = pd.get_dummies(df['city'], prefix='city')

df = pd.concat([df, city_dummies], axis=1)

df = df.drop(columns=['gender', 'city', 'smoker'])

df.head()

,age,blood_pressure,cholesterol,bmi,heart_disease,gender_encoded,smoker_encoded,city_Dammam,city_Jeddah,city_Makkah,city_Riyadh
0,56.0,171.7,295.2,34.4,1,1,0,False,False,False,True
1,69.0,112.4,345.2,21.9,0,0,0,True,False,False,False
2,46.0,126.9,253.3,39.9,0,0,0,False,True,False,False
3,32.0,158.0,214.6,37.7,1,0,0,False,False,False,True
4,60.0,110.6,309.0,40.8,1,1,1,False,False,True,False


Task 3

In [9]:
df.head(10)
df.shape

(200, 11)

Number of columns increased
One-hot creates multiple columns

PART 4: Feature Scaling


Step 1: Prepare Data

In [11]:
from sklearn.preprocessing import StandardScaler

feature_cols = [col for col in df.columns if col != 'heart_disease']

X = df[feature_cols].values
y = df['heart_disease'].values

Step 2: Scale

In [12]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Step 3: Compare

In [13]:
pd.DataFrame({
    'Original Mean': X.mean(axis=0),
    'Scaled Mean': X_scaled.mean(axis=0)
})

,Original Mean,Scaled Mean
0,49.345,7.549517e-17
1,136.0825,-4.207745e-16
2,252.361,1.332268e-16
3,29.326,-7.105427e-17
4,0.515,-2.664535e-17
5,0.32,-5.329071e-17
6,0.295,-8.881784e-18
7,0.26,-2.664535e-17
8,0.21,0.000000e+00
9,0.235,4.884981e-17


Task 4 (MinMaxScaler)

In [14]:
from sklearn.preprocessing import MinMaxScaler

scaler2 = MinMaxScaler()
X_minmax = scaler2.fit_transform(X)

PART 5: KNN Before vs After Scaling

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

Without Scaling

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

acc1 = accuracy_score(y_test, knn.predict(X_test))
print(acc1)

0.5


With Scaling

In [17]:
X_train_s, X_test_s, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

knn.fit(X_train_s, y_train)
acc2 = accuracy_score(y_test, knn.predict(X_test_s))

print(acc2)

0.4


Accuracy improves with scaling


KNN depends on distance

PART 6: Pipeline

In [18]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

pipe.fit(X_train, y_train)
pipe.predict(X_test)

array([1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1])

Task 6

In [19]:
from sklearn.impute import SimpleImputer

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

PART 7: Feature Selection

In [20]:
df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_scaled['heart_disease'] = y

corr = df_scaled.corr()
corr['heart_disease'].sort_values()

,heart_disease
city_Jeddah,-0.089100
city_Makkah,-0.061331
smoker_encoded,-0.025048
bmi,-0.004382
cholesterol,0.002672
gender_encoded,0.013705
city_Dammam,0.067146
city_Riyadh,0.078872
blood_pressure,0.106069
age,0.142702


Select important features

In [21]:
important = corr['heart_disease'][abs(corr['heart_disease']) > 0.05]
important

,heart_disease
age,0.142702
blood_pressure,0.106069
city_Dammam,0.067146
city_Jeddah,-0.089100
city_Makkah,-0.061331
city_Riyadh,0.078872
heart_disease,1.000000


Which feature strongest
Try different thresholds

FINAL PART


Data preprocessing had a significant impact on model performance, especially feature scaling. Scaling improved the accuracy of KNN because it ensures all features contribute equally to distance calculations. Handling missing values was also important because machine learning models cannot work with NaN values. Encoding categorical variables allowed the model to understand text data by converting it into numerical form. Without preprocessing, the model would perform poorly or fail completely. Overall, preprocessing is essential in real-world machine learning because data is often messy and unstructured. Proper preprocessing leads to better accuracy and more reliable models.